<a href="https://colab.research.google.com/github/smorgan-blip/lis4693/blob/main/lab-2/Lab_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2: Text Pre-Processing using spaCy

In this lab assignment, we will learn to perform some basic text pre-processing using spaCy.

*Note: Before starting this lab assignment, please complete the Introduction to spaCy notebook*

## Leaning Objectives

In this exercise, you will:

- Load and process your own text file (transcript.txt)
- Split text into sentences
- Count words and sentences
- Find frequently used words
- Use spaCy’s PhraseMatcher to find specific phrases
- Understand the difference between blank pipelines and full pipelines

This exercise builds directly on concepts from discussed in the precursor notebook on "Introduction to spaCy".

## Install spaCy

In [1]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 92.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import spacy

Here I am installing and importing spaCy which will let us create NLP models

## Load and read your file from GitHub


In [19]:
import requests

url = "https://raw.githubusercontent.com/smorgan-blip/lis4693/refs/heads/main/lab-1/transcript.txt"
response = requests.get(url)
response.raise_for_status() # Raise an exception for HTTP errors
text = response.text

Here I am inputting the text for this program from the transcript I generated in lab 1. It is stored in my GitHub so I found it in my directory, toggled on the raw mode, and linked the URL. Then we set the response from the request to the text.

In [21]:
print("Number of characters:", len(text)) # print number of characters

print(text[:100])   # print first 100 characters

Number of characters: 7968
Today on Sugar Spun Run I'll be showing you how to make Meringue Cookies.
Hey Sugar Spun Bakers, Sam


Here I found out using function not from an NLP model that there are 7968 characters in my transcript. I printed the first 100 characters below.

## Sentence Segmentation

Create a blank spaCy model and add sentencizer.

In [23]:
nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")

doc = nlp(text)

sentences = list(doc.sents)

print("Number of sentences:", len(sentences))

print("\nFirst 3 sentences:")
for sent in sentences[:3]:
    print(sent)

Number of sentences: 97

First 3 sentences:
Today on Sugar Spun Run I'll be showing you how to make Meringue Cookies.

Hey Sugar Spun Bakers, Sam here and today I'm going to be showing you one of my most popular and probably
even most unique cookie recipes on the blog.
This meringue cookie recipe has been around for a few years now.


Here we created the NLP model and fed the text from the URL to the doc. Then we used the NLP model's function to output the number of sentences and the text in sentence units. My transcript has 97 sentences and we printed the first 3 of them.

## Word Count and Token Analysis

Let's count total words and unique words in your text file.

In [24]:
words = [token.text.lower() for token in doc if token.is_alpha]

print("Total words:", len(words))
print("Unique words:", len(set(words)))

Total words: 1532
Unique words: 402


Here we found the number of words in my transcript was 1,532 and there were 402 unique words

## Most Frequent Words

Let's find top 10 most frequent words in your file.

In [28]:
from collections import Counter

word_freq = Counter(words)

print("Top 10 most frequent words:")
for word, count in word_freq.most_common(10):
    print(word, count)

Top 10 most frequent words:
to 85
you 57
the 40
and 37
a 37
of 32
i 31
are 30
your 28
that 26


Here my most frequent word was "to" and it makes sense because it is a preposition and my transcript does give directions and explain actions. The other most frequent words are also prepositions, articles, and pronouns and that makes sense because these words are always needed for composing English sentence structure despite the unique meanings the sentences could have.


## Using Full spaCy Pipeline

Now use the full model for better linguistic analysis.

In [26]:
nlp2 = spacy.load("en_core_web_sm")

doc2 = nlp2(text)

print("Named Entities:")
for ent in doc2.ents:
    print(ent.text, ent.label_)

Named Entities:
Today DATE
Meringue Cookies LOC
Sam PERSON
today DATE
one CARDINAL
a few years DATE
hundreds CARDINAL
5 CARDINAL
225 degrees QUANTITY
4 CARDINAL
today DATE
today DATE
one CARDINAL
first ORDINAL
a rainy day DATE
One CARDINAL
1/2 CARDINAL
1/8 CARDINAL
today DATE
KitchenAid ORG
1 cup QUANTITY
about 15 - 20 seconds DATE
next tablespoon TIME
about another minute TIME
another two minutes TIME
Atteco 846 PRODUCT
two CARDINAL
two CARDINAL
one CARDINAL
today DATE
only about 25 CARDINAL
225 degree QUANTITY
Fahrenheit ORG
one hour TIME
225 degree QUANTITY
Fahrenheit ORG
10 - 20 minutes TIME
another hour TIME


There were 38 entities identified in my transcript and the most frequent ones were dates, times, quantities, and cardinals. These make sense because the transcript is baking instructions and these quantitative descriptions are important in recipes.

## PhraseMatcher

In [30]:
from spacy.matcher import PhraseMatcher

matcher = PhraseMatcher(nlp2.vocab, attr="LOWER")

phrases = ["mix", "stir", "beat"]

patterns = [nlp2(p) for p in phrases]

matcher.add("TECH_TERMS", patterns)

matches = matcher(doc2)

print("Matches found:")
for match_id, start, end in matches:
    print(doc2[start:end])
    print("Sentence:", doc2[start].sent)

Matches found:
stir
Sentence: So we will keep our mixer running on high speed and we are going to continue to stir the sugar into
the egg mixture until it's dissolved.
beat
Sentence: Once you've added all of your sugar you are going to want to continue to beat until your meringue
is thick and shiny and has increased in volume.
beat
Sentence: Mine is not completely incorporated at this point, so we are going to continue to beat it until the sugar's dissolved.

stir
Sentence: This is just a teaspoon of vanilla, and I'll stir it in now.

mix
Sentence: You could mix this in by hand, I'm just going to mix it in with my mixer on low speed.

mix
Sentence: You could mix this in by hand, I'm just going to mix it in with my mixer on low speed.

beat
Sentence: It is possible to over-beat meringue, so you don't want to
push it once you get to that point where your sugars dissolved and your mixture is nice and thick and glossy.



The phrase matcher here identifies and prints instances of phrases. I used the phrases "mix", "stir", and "beat" because I thought they would show up in the baking instructions and they did

REFLECTION: The first time I did this lab I completely messed it up. I thought we were supposed to use the United States text and encountered my first problem where the URL was incorrect. It was giving me errors and I knew if it couldn't access the text nothing would work. So I went through your GitHub to find the new text, toggled on the raw mode, and got the new link for the command. That was when I found out that we were supposed to use our original transcripts from the first lab, but because I had figured out how to find the raw text data from GitHub, I was able to go to my own page and find the link to my own text. After that, the assignment was very easy and straightforward. I followed the directions and code to see which commands I had to adjust to get the proper output and successfully got the data from my transcript using the NLP model.

# EXCERCISE

Open a new Google Colab notebook and complete the tasks below. As you work, add brief explanations using the **Text (Markdown) cells** throughout your notebook to describe what you are doing.

1. Make a new folder named `lab-2` in your `lis4693` or `lis5693` repo on GitHub. **[1 Point]**

2. Complete the following tasks:


**TASK 1**: Load and read your `transcript.txt` file from lab-1 from GitHub repo directly **[1 Point]**

**TASK 2**: How many characters are in your transcript? Print the first 100 characters. **[1 Point]**

**TASK 3**: Perform sentence segmentation using the blank pipeline
  - How many sentences are in your transcript? **[0.5 Point]**
  - Print the first 3 sentences **[0.5 Point]**

**TASK 4**: Perform Word Count and Token Analysis
  - What is the total number of words? **[0.5 Point]**
  - What is the number of unique words? **[0.5 Point]**

**TASK 5**: Find Most Frequent Words
  - What is the most frequent word? **[0.5 Point]**
  - Why do you think this word appears frequently? **[1 Point]**

**TASK 6**: Run full spaCy pipeline
  - How many named entities were found? **[0.5 Point]**
  - What types of entities appear? (PERSON, ORG, DATE, etc.) **[0.5 Point]**

**TASK 7**: Use PhraseMatcher **[1 Point]**

When you selected your YouTube video in Lab-1, what topic or subject were you interested in? Based on that topic, identify three specific phrases that are directly relevant to your search. For example, if your video was about information retrieval, relevant phrases might include "information retrieval," "text mining," and "data science."

Make sure the video you selected in Lab-1 was clearly related to your chosen topic and was the same video for which you downloaded the transcript in Lab-1.

**TASK 8**: At the end of your Colab notebook, create a new text cell and write a brief reflection for this assignment in a few sentences addressing the following **[2 Points]**:
 - What went well?
 - What did not go well or what challenges you encountered?

3. Push your Google Colab file to your `lab-2` GitHub repo from Colab. *No points will be given if you upload it to GitHub directly!* **[1 Point]**
4. Share the link to your `lab-2` GitHub repo for this lab assignment on CANVAS for credit. **[0.5 Point]**
